# Bronze Orchestrator: sell_in
Orchestrates ingestion, validation, and monitoring for the Bronze layer of sell_in.

**Execution Order:**
1. Ingestion
2. Validation
3. Monitoring

**Alerts:**
- Execution errors are captured and displayed for each step.
- If any step fails, subsequent steps are not executed.

In [ ]:
# Install openpyxl for Excel ingestion
%pip install openpyxl

import sys
sys.path.append("/Workspace/Users/diego.mayorgacapera@gmail.com/.bundle/BI_Market_Visibility/dev/files")
# Import orchestrated functions
from src.bronze.sell_in.ingest_sell_in import run_ingestion
from src.bronze.sell_in.validate_sell_in import run_validation
from src.bronze.sell_in.monitor_sell_in import run_monitoring

In [ ]:
# Force reload of updated modules to ensure notebook uses latest code from bundle
import importlib
import src.bronze.sell_in.ingest_sell_in as ingest_sell_in_mod
importlib.reload(ingest_sell_in_mod)
run_ingestion = ingest_sell_in_mod.run_ingestion

In [ ]:
# --- Environment Parameters ---
# You can change these for staging/prod
# Bronze table and validation table names must match your deployment
# Example: env = 'prod'
# Example: bronze_table = 'workspace.bronze.sell_in'
# Example: validation_table = 'workspace.bronze.sell_in_validation'
env = 'dev'
bronze_table = 'workspace.bronze.sell_in'
validation_table = 'workspace.bronze.sell_in_validation'

In [ ]:
# Step 1: Ingestion
try:
    batch_id, rows_ingested = run_ingestion(source_path='/Volumes/workspace/raw_data/sell_in', delta_table=bronze_table, env=env, dbutils=dbutils)
    print(f'✅ Ingestion completed. Batch ID: {batch_id}, Rows: {rows_ingested}')
except Exception as e:
    print(f'❌ Ingestion failed: {str(e)}')
    raise

In [ ]:
# Step 2: Validation (only if ingestion succeeded and rows_ingested > 0)
if rows_ingested == 0 or batch_id is None:
    print('⚠️ No new data ingested. Skipping validation.')
    metrics_df = None
else:
    try:
        metrics_df = run_validation(bronze_table=bronze_table, batch_id=batch_id, env=env)
        print('✅ Validation completed.')
        display(metrics_df)
    except Exception as e:
        print(f'❌ Validation failed: {str(e)}')
        raise

In [ ]:
# Step 3: Monitoring (only if validation succeeded and rows_ingested > 0)
if rows_ingested == 0 or batch_id is None:
    print('⚠️ No new data ingested. Skipping monitoring.')
    metrics_json, alerts_json = [], []
else:
    try:
        metrics_json, alerts_json = run_monitoring(bronze_table=bronze_table, validation_table=validation_table, batch_id=batch_id, env=env)
        print('✅ Monitoring completed.')
        print('Metrics:')
        print(metrics_json)
        print(f'Metrics volume: {len(metrics_json)}')
        if alerts_json:
            print('⚠️ Alerts:')
            print(alerts_json)
            print(f'Alerts volume: {len(alerts_json)}')
        else:
            print('No alerts triggered.')
    except Exception as e:
        print(f'❌ Monitoring failed: {str(e)}')
        raise

In [ ]:
# --- Logging setup ---
import logging
logger = logging.getLogger("bronze_orchestrator")
if not logger.hasHandlers():
    logger.setLevel(logging.INFO)
    handler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s %(levelname)s %(name)s: %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)